# Laboratorio 7
Alina Carías, Daniel Machic y Ariela Mishaan

**Github:** https://github.com/ArielaMishaanCohen/LAB7.git 

## Task 1

## Task 2

### Pregunta 2.1

"Para el modelo de radiografías no tiene sentido usar Transfer Learning desde ImageNet; ese dataset tiene fotos de perros, gatos y autos. Las radiografías son imágenes en escala de grises totalmente distintas. Mejor entrenamos desde cero para no contaminar el modelo.”

**1. ¿Está de acuerdo con el desarrollador? Argumentar con base en lo que las capas convolucionales tempranas de una CNN realmente aprenden y por qué ese conocimiento sí es transferible, incluso entre dominios visualmente distintos.**  

No estoy de acuerdo con el desarrollador. Aunque ImageNet sí tiene imágenes muy distintas, las primeras capas de una CNN no aprenden objetos específicos, sino patrones básicos visuales como bordes, texturas, gradientes de intensidad y formas simples. Estos patrones también se pueden ver en las radiografías médicas (por ejemplo un borde puede representar la orilla de un pulmón, o una textura podría mostrar la condición del tejido). Dado que las capas tempranas aprenden estas características universales, el conocimiento de ImageNet sí es transferible, aunque sea en temas distintos. Entrenar desde cero implicaría que el modelo tenga que volver a entender estos patrones básicos, lo cual es ineficiente para solo 800 imágenes. 

**2. Más allá del argumento técnico, ¿cuál es el costo real de entrenar desde cero para un startup como MediScan Guatemala? Considerar al menos dos dimensiones distintas a la puramente computacional (ej. tiempo al mercado, riesgo, talento humano).**  

Entrenar desde cero no solo es más caro computacionalemente sino también implica otros contratiempos: 

- **Tiempo al mercado:** se necesitarían muchas más épocas y experimentación de los modelos. Esto retrasa la implementación del sistema en las clínicas. En el contexto médico, esto significa pacientes sin apoyo diagnóstico por más tiempo. 
- **Riesgo técnico:** con pocos datos, el modelo probablemente tenga sobreajuste. Esto da un alto riesgo de obtener un modelo poco confiable, que en este contexto es peligroso. 
- **Talento humano:** entrenar desde cero requiere más experiencia para ajustar la arquitectura e hiperparámetros desde cero. 

**3. ¿Existe algún escenario legítimo en el que usted sí consideraría entrenar desde cero en lugar de hacer transfer learning?**  

Por ejemplo, si se tuviera un dataset muy grande y específico del dominio médico (por ejemplo decenas de miles de radiografías), podría valer la pena entrenar desde cero. Otro caso es en el que el dominio sea muy diferente estructuralmente (imágenes demasiado distintas como MRIs en 3D). Si el uso de transfer learning resultara en sesgos o resultados no óptimos, se tendría que evaluar la opción también. 


### Pregunta 2.2

Equipo debe decidir estrategia de Fine Tuning.  

**4. Dada la naturaleza del dataset médico disponible, ¿cuál opción recomendaría y por qué?**  

Creo que la mejor estrategia sería una combinada. Pero si hubiera que escoger entre una de las dos, creo que la opción A es la mejor, es decir, congelar toda la red base. La razón es que dado nuestro dataset tan pequeño, existe un riesgo muy alto de sobreajuste y si se descongelan todas las capas, se podría perder la capacidad de generalización. Además, el modelo perdería la capacidad de reconocer patrones básicos como bordes y texturas. En la opción A, en cambio, se aprovechan las features ya aprendidas y se reduce el riesgo del sobreajuste. 

**5. ¿Qué riesgo específico correría si aplica la opción B con una taza de aprendizaje alta (ej. 1e-3) en toda la red?**  

El riesgo principal de usar el B con tasa de aprendizaje alta es el fenómeno que se llama "olvido catastrófico". Esto significa que el modelo pierde rápidamente los patrones útiles qeu aprendió en ImageNet, adaptándose demasiado rápido a un dataset pequeño. Al final aprende solo ruido o patrones no generalizables. Durante el entrenamiento, esto se vería como una pérdida que baja demasiado rápida (en train), mientras que la pérdida de la validación sube o varía demasiado. Los resultabels del entrenamiento serían  inestables.  

**6. Si con el tiempo MediScan Guatemala logra recolectar 50000 radiografías etiquetadas, ¿cambiaría su recomendación?**  

Sí. Si se tuvieran muchos más datos, sería más útil la opción B de descongelar más capas, pero manteniendo un learning rate bajo en las capas iniciales y uno más alto en el cabezal final. Esto es porque ya tendríamos suficientes datos para reducir el riesgo del sobreajuste y permitir que el modelo aprenda más patrones específicos deel ambito médico que se está analizando. 

### Pregunta 2.3

Requerimiento: expandir sistema para que también detecte fracturas en radiografías de muñeca, usando el mismo modelo que se tiene para la neumonía. ¿Se puede reutilizar directamente o se tiene que empezar todo de nuevo?  

**7. Evaluar la viabilidad de reutilizar el modelo de neumonía para el nuevo problema de fracturas en al muñeca. ¿Qué partes del modelo podrían aprovecharse y cuáles habría que modificar?**  

Considero que sí es viable reutilizar el modelo, pero con algunos cambios. Se podrían aprovechar las capas tempranas, que aprenden patrones generales como bordes, texturas y formas. Sin embargo, lo que sí valdría la pena modificar son las últimas capas, que se especializan en detectar neumonía en pulmones, y no sirven directamente para detectar fracturas. El nuevo cabezal de clasificación cambiaría completamente, porque ahora tendría que clasificar entre fractura y no fractura.  

**8. Proponer un plan de acción concreto de 3 pasos para adaptar el modelo exitente al nuevo dominio. Ser específico: ¿Qué congela, qué descongela, qué modifica en el cabezal, qué tasa de aprendizaje usaría y por qué?**  

- **Reemplazar el cabezal:** cambiar la capa final por una nueva. Ahora la salida binaria sería fractura/no fractura. El learning rate para esta capa sería 1e-3. 
- **Congelar capas tempranas:** congelar las primeras capas (features generales). Entrenar solo las últimas capas y el cabezal. La base queda congelada. 
- **Fine-tuning gradual:** descongelar las últimas capas progresivamente. Ajustar con learning rate bajo (1e-5) para adaptar el modelo al nuevo dominio sin perder lo que se aprendió antes.  

**9. ¿Qué riesgo ético o clínico implica reutilizar un modelo entrenado en un dominio para otro dominio médico sin una validación rigurosa? ¿Qué proceso de validación mínimo recomendaría antes de desplegarlo?**  

El riesgo principal de reutilizar el modelo sin validación puede llevar a diagnósticos incorrectos (falsos negativos o positivos), pérdida de confianza médica y daño al paciente. La validación mínima recomendada sería que antes de desplegar el sistema, se pruebe con fotos que nunca se vieron durante el entrenamiento para volver a evaluar el rendimiento con métricas relevantes (como recall, F1-score, precision - no solo accuracy). Además, recomendaría evaluación humana de expertos antes de desplegar el sistema y también probarlo en distintos hospitales y con distintos equipos para asegurar la generalización. 